In [5]:
import json
import pandas as pd

with open("../../mock_data.json", "r") as f:
    data = json.load(f)
print(data.keys())
# vulnerabilities = data["vulnerabilities"]

# df = pd.DataFrame(vulnerabilities)

# df

dict_keys(['schema_version', 'assets', 'vulnerabilities', 'network_connections', 'users', 'risk_metrics', 'controls', 'optimization_results'])


In [6]:
vulnerabilities = data["vulnerabilities"]

print(len(vulnerabilities))

5


In [7]:
df = pd.DataFrame(vulnerabilities)

df

,cve_id,name,cvss_score,severity,cisa_kev,exploit_age_days,exploit_probability,description
0,CVE-2021-44228,Log4Shell Remote Code Execution,10.0,CRITICAL,True,1005,0.95,Apache Log4j2 JNDI features used in configurat...
1,CVE-2023-34362,MOVEit Transfer SQL Injection RCE,9.8,CRITICAL,True,468,0.88,Unauthenticated SQL injection vulnerability in...
2,CVE-2023-23397,Microsoft Outlook NTLM Privilege Escalation,9.8,CRITICAL,True,540,0.82,Microsoft Outlook elevation of privilege vulne...
3,CVE-2024-30078,Windows Wi-Fi Driver Remote Code Execution,8.8,HIGH,False,90,0.64,An unauthenticated attacker could execute remo...
4,CVE-2024-21626,runc Container Breakout Leaky File Descriptors,8.6,HIGH,True,220,0.76,Container escape vulnerability in runc allowin...


In [8]:
df[
    [
        "cve_id",
        "cvss_score",
        "cisa_kev",
        "exploit_age_days",
        "exploit_probability"
    ]
]

,cve_id,cvss_score,cisa_kev,exploit_age_days,exploit_probability
0,CVE-2021-44228,10.0,True,1005,0.95
1,CVE-2023-34362,9.8,True,468,0.88
2,CVE-2023-23397,9.8,True,540,0.82
3,CVE-2024-30078,8.8,False,90,0.64
4,CVE-2024-21626,8.6,True,220,0.76


In [9]:
X = df[["cvss_score", "cisa_kev", "exploit_age_days",
        "exploit_probability"]]   # ❌

In [10]:
features = df[
    [
        "cvss_score",
        "cisa_kev",
        "exploit_age_days"
    ]
].copy()

features

,cvss_score,cisa_kev,exploit_age_days
0,10.0,True,1005
1,9.8,True,468
2,9.8,True,540
3,8.8,False,90
4,8.6,True,220


In [11]:
features["cisa_kev"] = features["cisa_kev"].astype(int)

features

,cvss_score,cisa_kev,exploit_age_days
0,10.0,1,1005
1,9.8,1,468
2,9.8,1,540
3,8.8,0,90
4,8.6,1,220


In [14]:
X = features


In [15]:
df[
    [
        "cve_id",
        "exploit_probability"
    ]
]

,cve_id,exploit_probability
0,CVE-2021-44228,0.95
1,CVE-2023-34362,0.88
2,CVE-2023-23397,0.82
3,CVE-2024-30078,0.64
4,CVE-2024-21626,0.76


In [16]:
import numpy as np

np.random.seed(42)

n = 500

synthetic_data = pd.DataFrame({
    "cvss_score": np.random.uniform(3.0, 10.0, n),
    "cisa_kev": np.random.randint(0, 2, n),
    "exploit_age_days": np.random.randint(1, 1000, n)
})

synthetic_data.head()

,cvss_score,cisa_kev,exploit_age_days
0,5.621781,1,947
1,9.655000,0,198
2,8.123958,0,646
3,7.190609,0,579
4,4.092130,0,391


In [18]:
synthetic_data.head(10)

,cvss_score,cisa_kev,exploit_age_days
0,5.621781,1,947
1,9.655000,0,198
2,8.123958,0,646
3,7.190609,0,579
4,4.092130,0,391
5,4.091962,0,179
6,3.406585,1,968
7,9.063233,0,554
8,7.207805,1,448
9,7.956508,1,500


In [19]:
score = (
    0.35 * (synthetic_data["cvss_score"] / 10)
    + 0.45 * synthetic_data["cisa_kev"]
    + 0.20 * (synthetic_data["exploit_age_days"] / 1000)
)

score = np.clip(score, 0, 1)

synthetic_data["synthetic_probability"] = score

synthetic_data.head()

,cvss_score,cisa_kev,exploit_age_days,synthetic_probability
0,5.621781,1,947,0.836162
1,9.655000,0,198,0.377525
2,8.123958,0,646,0.413539
3,7.190609,0,579,0.367471
4,4.092130,0,391,0.221425


In [20]:
synthetic_data["exploited"] = (
    np.random.random(n) < synthetic_data["synthetic_probability"]
).astype(int)

synthetic_data.head(10)

,cvss_score,cisa_kev,exploit_age_days,synthetic_probability,exploited
0,5.621781,1,947,0.836162,1
1,9.655000,0,198,0.377525,0
2,8.123958,0,646,0.413539,0
3,7.190609,0,579,0.367471,0
4,4.092130,0,391,0.221425,1
5,4.091962,0,179,0.179019,1
6,3.406585,1,968,0.762830,1
7,9.063233,0,554,0.428013,0
8,7.207805,1,448,0.791873,1
9,7.956508,1,500,0.828478,1


In [21]:
synthetic_data["exploited"].value_counts()

exploited
1    286
0    214
Name: count, dtype: int64

In [22]:
X_train_data = synthetic_data[
    [
        "cvss_score",
        "cisa_kev",
        "exploit_age_days"
    ]
]

y_train_data = synthetic_data["exploited"]

In [23]:
print("X shape:", X_train_data.shape)
print("y shape:", y_train_data.shape)

X shape: (500, 3)
y shape: (500,)


In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_train_data,
    y_train_data,
    test_size=0.2,
    random_state=42,
    stratify=y_train_data
)

In [26]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (400, 3)
X_test: (100, 3)
y_train: (400,)
y_test: (100,)


In [28]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

In [29]:
model.fit(
    X_train,
    y_train
)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [30]:
y_pred = model.predict(X_test)

print(y_pred[:20])

[1 1 1 0 0 1 0 1 1 1 1 0 0 1 1 1 1 0 1 0]


In [31]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.95


In [37]:
y_probability = model.predict_proba(X_test)

print(y_probability[:10])

[[4.2856336e-03 9.9571437e-01]
 [2.3569465e-03 9.9764305e-01]
 [1.5706718e-02 9.8429328e-01]
 [9.9945229e-01 5.4769043e-04]
 [9.0996134e-01 9.0038650e-02]
 [1.7222345e-02 9.8277766e-01]
 [9.9863470e-01 1.3653206e-03]
 [9.6762180e-04 9.9903238e-01]
 [1.5017986e-03 9.9849820e-01]
 [2.2410154e-03 9.9775898e-01]]


In [38]:
p_exploit = model.predict_proba(X_test)[:, 1]

print(p_exploit[:10])

[9.9571437e-01 9.9764305e-01 9.8429328e-01 5.4769043e-04 9.0038650e-02
 9.8277766e-01 1.3653206e-03 9.9903238e-01 9.9849820e-01 9.9775898e-01]


In [39]:
results = X_test.copy()

results["actual"] = y_test.values
results["predicted"] = y_pred
results["p_exploit"] = p_exploit

results.head(10)

,cvss_score,cisa_kev,exploit_age_days,actual,predicted,p_exploit
452,5.969554,1,296,1,1,0.995714
185,7.619888,1,476,1,1,0.997643
479,6.661685,1,17,1,1,0.984293
118,9.247913,0,182,0,0,0.000548
386,4.974242,0,168,0,0,0.090039
400,3.721867,0,558,1,1,0.982778
352,7.610855,0,4,0,0,0.001365
205,3.064379,1,429,1,1,0.999032
235,8.057165,1,979,1,1,0.998498
164,3.632028,0,710,1,1,0.997759


In [40]:
real_features = df[
    [
        "cvss_score",
        "cisa_kev",
        "exploit_age_days"
    ]
].copy()

real_features["cisa_kev"] = real_features["cisa_kev"].astype(int)

real_probabilities = model.predict_proba(real_features)[:, 1]

print(real_probabilities)

[0.193569   0.16341938 0.16341938 0.00065296 0.20328346]


In [41]:
comparison = df[
    [
        "cve_id",
        "cvss_score",
        "cisa_kev",
        "exploit_age_days",
        "exploit_probability"
    ]
].copy()

comparison["xgboost_probability"] = real_probabilities

comparison

,cve_id,cvss_score,cisa_kev,exploit_age_days,exploit_probability,xgboost_probability
0,CVE-2021-44228,10.0,True,1005,0.95,0.193569
1,CVE-2023-34362,9.8,True,468,0.88,0.163419
2,CVE-2023-23397,9.8,True,540,0.82,0.163419
3,CVE-2024-30078,8.8,False,90,0.64,0.000653
4,CVE-2024-21626,8.6,True,220,0.76,0.203283


In [42]:
print(synthetic_data["exploited"].value_counts())

exploited
1    286
0    214
Name: count, dtype: int64


In [43]:
print(synthetic_data["synthetic_probability"].describe())

count    500.000000
mean       0.539082
std        0.248471
min        0.113043
25%        0.316818
50%        0.476273
75%        0.775501
max        0.984284
Name: synthetic_probability, dtype: float64


In [44]:
synthetic_data.sort_values(
    "synthetic_probability",
    ascending=False
).head(10)

,cvss_score,cisa_kev,exploit_age_days,synthetic_probability,exploited
154,9.899553,1,939,0.984284,0
478,9.785118,1,957,0.983879,1
395,9.515301,1,922,0.967436,1
441,9.906477,1,849,0.966527,0
471,9.789152,1,866,0.965820,0
390,9.933536,1,821,0.961874,0
252,8.957957,1,984,0.960328,1
228,9.244326,1,925,0.958551,1
248,9.745340,1,833,0.957687,0
364,9.590254,1,853,0.956259,1


In [45]:
print("Number of exploited:", y_train_data.sum())
print("Number of not exploited:", len(y_train_data) - y_train_data.sum())

Number of exploited: 286
Number of not exploited: 214


In [46]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 500

synthetic_data = pd.DataFrame({
    "cvss_score": np.random.uniform(3.0, 10.0, n),
    "cisa_kev": np.random.randint(0, 2, n),
    "exploit_age_days": np.random.randint(1, 1000, n)
})

synthetic_data.head()

,cvss_score,cisa_kev,exploit_age_days
0,5.621781,1,947
1,9.655000,0,198
2,8.123958,0,646
3,7.190609,0,579
4,4.092130,0,391


In [47]:
score = (
    0.35 * (synthetic_data["cvss_score"] / 10)
    + 0.45 * synthetic_data["cisa_kev"]
    + 0.20 * (synthetic_data["exploit_age_days"] / 1000)
)

score = np.clip(score, 0, 1)

synthetic_data["synthetic_probability"] = score

synthetic_data.head()

,cvss_score,cisa_kev,exploit_age_days,synthetic_probability
0,5.621781,1,947,0.836162
1,9.655000,0,198,0.377525
2,8.123958,0,646,0.413539
3,7.190609,0,579,0.367471
4,4.092130,0,391,0.221425


In [48]:
synthetic_data["exploited"] = (
    np.random.random(n) < synthetic_data["synthetic_probability"]
).astype(int)

synthetic_data.head()

,cvss_score,cisa_kev,exploit_age_days,synthetic_probability,exploited
0,5.621781,1,947,0.836162,1
1,9.655000,0,198,0.377525,0
2,8.123958,0,646,0.413539,1
3,7.190609,0,579,0.367471,0
4,4.092130,0,391,0.221425,0


In [49]:
X_train_data = synthetic_data[
    [
        "cvss_score",
        "cisa_kev",
        "exploit_age_days"
    ]
]

y_train_data = synthetic_data["exploited"]

print("X shape:", X_train_data.shape)
print("y shape:", y_train_data.shape)

X shape: (500, 3)
y shape: (500,)


In [50]:
X_train, X_test, y_train, y_test = train_test_split(
    X_train_data,
    y_train_data,
    test_size=0.2,
    random_state=42,
    stratify=y_train_data
)

In [51]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

In [52]:
model.fit(X_train, y_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [53]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.67


In [54]:
real_features = df[
    [
        "cvss_score",
        "cisa_kev",
        "exploit_age_days"
    ]
].copy()

real_features["cisa_kev"] = real_features["cisa_kev"].astype(int)

real_probabilities = model.predict_proba(
    real_features
)[:, 1]

print(real_probabilities)

[0.96070796 0.98056316 0.9820983  0.6017991  0.9533146 ]


In [55]:
comparison = df[
    [
        "cve_id",
        "cvss_score",
        "cisa_kev",
        "exploit_age_days",
        "exploit_probability"
    ]
].copy()

comparison["xgboost_probability"] = real_probabilities

comparison

,cve_id,cvss_score,cisa_kev,exploit_age_days,exploit_probability,xgboost_probability
0,CVE-2021-44228,10.0,True,1005,0.95,0.960708
1,CVE-2023-34362,9.8,True,468,0.88,0.980563
2,CVE-2023-23397,9.8,True,540,0.82,0.982098
3,CVE-2024-30078,8.8,False,90,0.64,0.601799
4,CVE-2024-21626,8.6,True,220,0.76,0.953315


In [56]:
comparison[
    [
        "cve_id",
        "exploit_probability",
        "xgboost_probability"
    ]
]

,cve_id,exploit_probability,xgboost_probability
0,CVE-2021-44228,0.95,0.960708
1,CVE-2023-34362,0.88,0.980563
2,CVE-2023-23397,0.82,0.982098
3,CVE-2024-30078,0.64,0.601799
4,CVE-2024-21626,0.76,0.953315


In [57]:
print(real_probabilities)

[0.96070796 0.98056316 0.9820983  0.6017991  0.9533146 ]


In [58]:
import os

os.makedirs("../models", exist_ok=True)

model.save_model("../models/exploit_model.json")

print("Model saved successfully!")

Model saved successfully!


In [59]:
import os

print(os.path.exists("../models/exploit_model.json"))

True
